# Training phase

Trains each of the 6 policies (3 discrete tabular methods, 3 continuous methods) with fixed hyperparameters and saves each one to `saved_policies/<name>/`. Run cells individually per algorithm, or run the whole notebook top to bottom.

Model testing / success-criterion evaluation lives in `test_models.ipynb`, not here.

In [ ]:
import sys
!git clone https://github.com/NomeMio/rl_inverse_pendulum.git

sys.path.insert(0, "/content/rl_inverse_pendulum")
    
!pip install -r /content/rl_inverse_pendulum/requirements.txt
!pip install gymnasium[mujoco]

In [1]:
from cart_model import Cart_model
from discrete_policies import SarsaAgent, QLearningAgent, ExpectedSarsaAgent
from continuous_policies import ActorPolicyContinuousSpace, ReinforcePolicy, SarsaTileCoding
from train import train_step_policy, train_episodic_policy

context = Cart_model(human=False)

/usr/local/anaconda3/envs/light_rl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## SARSA (discrete)

In [ ]:
sarsa_policy = SarsaAgent(gamma=0.98, alpha=0.1, epsilon_start=0.5, epsilon_min=0.05,
                           n_state_buckets=7, n_action_buckets=9)
train_step_policy(sarsa_policy, context, episodes=8000, max_steps=1000, desc="sarsa")
sarsa_policy.save("saved_policies/sarsa")

## Q-learning (discrete)

In [ ]:
q_learning_policy = QLearningAgent(gamma=0.98, alpha=0.1, epsilon_start=0.5, epsilon_min=0.05,
                                    n_state_buckets=7, n_action_buckets=9)
train_step_policy(q_learning_policy, context, episodes=8000, max_steps=1000, desc="q_learning")
q_learning_policy.save("saved_policies/q_learning")

## Expected SARSA (discrete)

In [ ]:
expected_sarsa_policy = ExpectedSarsaAgent(gamma=0.98, alpha=0.1, epsilon_start=0.5, epsilon_min=0.05,
                                            n_state_buckets=7, n_action_buckets=9)
train_step_policy(expected_sarsa_policy, context, episodes=8000, max_steps=1000, desc="expected_sarsa")
expected_sarsa_policy.save("saved_policies/expected_sarsa")

## Actor-Critic (continuous) — grid search over alpha_w, alpha_rho, discount

In [ ]:
actor_critic_best_params, actor_critic_policy, actor_critic_best_metrics, actor_critic_search_results = (
    ActorPolicyContinuousSpace.grid_search(
        context,
        train_episodes=2000,       # per-combo search budget (smaller than a full training run)
        max_steps=1000,
        eval_episodes=50,
        final_train_episodes=50000,  # retrain the winning combo from scratch at this larger budget
        desc="actor_critic",
    )
)
print("actor_critic best params:", actor_critic_best_params)
print("actor_critic best metrics:", actor_critic_best_metrics)
actor_critic_policy.save("saved_policies/actor_critic")

## REINFORCE (continuous)

In [ ]:
reinforce_policy = ReinforcePolicy(alpha_rho=0.005, discount=0.99)
train_episodic_policy(reinforce_policy, context, episodes=5000, max_steps=1000, desc="reinforce")
reinforce_policy.save("saved_policies/reinforce")

## Semi-gradient SARSA with tile coding (continuous state, discretized action)

In [ ]:
sarsa_tile_coding_policy = SarsaTileCoding(n_tilings=8, tiles_per_dim=6, alpha=0.1, gamma=0.98,
                                            epsilon_start=0.3, epsilon_min=0.02, n_action_buckets=9)
train_step_policy(sarsa_tile_coding_policy, context, episodes=5000, max_steps=1000, desc="sarsa_tile_coding")
sarsa_tile_coding_policy.save("saved_policies/sarsa_tile_coding")

In [ ]:
context.close()